# GraphRAG: page inputs and the native graph index

GraphRAG Standard Indexing extracts entities and relationships, groups the graph
into communities, generates community reports and stores the associated embeddings.
The index retains the prepared page inputs used by Classical RAG.

See the [GraphRAG workflow](../docs/illustrations/graphrag-workflow.pdf)
for indexing, Local Search and source-page evaluation.

Run `uv sync --locked --extra pdf --extra graphrag --extra evaluation` and start Jupyter with
`uv run --extra pdf --extra graphrag --extra evaluation jupyter lab`. Read [Data and evidence](01_data_and_evidence.ipynb)
first. Default execution inspects the saved public RFC 2795 index without model requests.
New indexing is explicitly selected and writes to ignored `local/`.
See thesis Section 2.5 and Appendix F for method details.

The same native path supports the five GraphRAG conditions in
[controlled changes](04_controlled_changes.ipynb). Choose their `CONDITION_ID`,
matching prepared inputs and a distinct ignored output folder. New queries use
the Local Search notebook, which reads the condition from the saved index.


In [1]:
import hashlib
import importlib.metadata
import json
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd
import yaml
from IPython.display import Markdown, display
from tabulate import tabulate

from rag_comparison import config
from rag_comparison.artifacts import verify_artifacts
import shutil

## Inputs and execution mode

The input-only manifest from notebook 01 supplies chunks, origins and questions.
For a new public run, select that prepared folder and set `RUN_AZURE_INDEX = True`.
For private inputs use an ignored executed notebook copy and a distinct run folder.
The default mode reads the version-bound public index manifest instead.

In [2]:
CONDITION_ID = "G0"
condition = config.CONDITIONS[CONDITION_ID]
assert condition["system"] == "G"
embedding_model = condition["embedding_model"]
embedding_variable = config.EMBEDDING_DEPLOYMENT_VARIABLES[embedding_model]
source_index_dir = Path("../examples/graphrag")
RUN_AZURE_INDEX = False
prepared_dir = Path("../local/classical_rag/rfc2795-inputs")
azure_dir = Path("../local/graphrag/rfc2795-azure-example")
data_dir = Path("../examples/graphrag")
if RUN_AZURE_INDEX:
    input_manifest = json.loads(
        (prepared_dir / "run_manifest.json").read_text(encoding="utf-8")
    )
    bound_paths = verify_artifacts(prepared_dir, input_manifest)
else:
    index_manifest = json.loads(
        (data_dir / "index_manifest.json").read_text(encoding="utf-8")
    )
    assert index_manifest["graphrag_version"] == config.GRAPHRAG_VERSION
    bound_paths = verify_artifacts(data_dir, index_manifest)
assert importlib.metadata.version("graphrag") == config.GRAPHRAG_VERSION
if RUN_AZURE_INDEX:
    assert (
        input_manifest.get("chunking_policy", config.PAGE_CHUNKING_POLICY)
        == condition["chunking_policy"]
    )
else:
    assert index_manifest["condition_id"] == CONDITION_ID

## Preserve the page identity

Each baseline input record uses `graphrag_input::<chunk_id>` as its native document ID.
GraphRAG JSONL input preserves this ID when `id_column` is set. Its generated
document title is a display label; the input ledger supplies the PDF page identity.

Structure inputs keep their original stable segment IDs, with the same original page identity supplied by the ledger.

In [3]:
def read_jsonl(path: Path) -> list[dict]:
    return [
        json.loads(line)
        for line in path.read_text(encoding="utf-8").splitlines()
        if line
    ]


chunks = read_jsonl(bound_paths["chunks"])
origins = read_jsonl(bound_paths["chunk_origins"])
questions = read_jsonl(bound_paths["questions"])
assert [row["chunk_id"] for row in chunks] == [row["chunk_id"] for row in origins]
inputs = [
    {
        "input_record_id": (
            row["chunk_id"]
            if condition["chunking_policy"] == config.STRUCTURE_POLICY["POLICY_ID"]
            else f"graphrag_input::{row['chunk_id']}"
        ),
        "text": row["text"],
    }
    for row in chunks
]
ledger = [
    {
        "target_artifact_id": row["input_record_id"],
        "source_evidence_unit_ids": origin["origin_evidence_unit_ids"],
        "projection_status": origin["projection_status"],
    }
    for row, origin in zip(inputs, origins, strict=True)
]
assert len({row["input_record_id"] for row in inputs}) == len(inputs)
assert all(
    row["projection_status"] == "projected"
    and len(row["source_evidence_unit_ids"]) == 1
    for row in ledger
)
print({"input_records": len(inputs), "questions": len(questions)})

{'input_records': 20, 'questions': 4}


## Azure model settings

The native provider is Azure for both model groups. Completion uses `gpt-4.1-mini`;
embedding uses the selected condition's small or large model at 1536 dimensions. Access values and deployment names remain
environment placeholders. The empty completion `call_args` dictionary keeps the
native GraphRAG contract; Classical-RAG decoding is a separate configuration.

In [4]:
settings = {
    "concurrent_requests": (
        config.GRAPHRAG_ENTITY_CONCURRENT_REQUESTS
        if CONDITION_ID == "G-ENTITY-POLICY-PROCESS-ROLE"
        else config.GRAPHRAG_CONCURRENT_REQUESTS
    )
}
model_groups = [
    (
        "completion_models",
        "default_completion_model",
        config.GENERATION_MODEL,
        "AZURE_OPENAI_CHAT_DEPLOYMENT",
    ),
    (
        "embedding_models",
        "default_embedding_model",
        embedding_model,
        embedding_variable,
    ),
]
for group, name, model, deployment_variable in model_groups:
    settings[group] = {
        name: {
            "model_provider": "azure",
            "model": model,
            "azure_deployment_name": "${" + deployment_variable + "}",
            "api_key": "${AZURE_OPENAI_API_KEY}",
            "api_base": "${AZURE_OPENAI_ENDPOINT}",
            "api_version": "${AZURE_OPENAI_API_VERSION}",
            "retry": {"type": "exponential_backoff"},
        }
    }
if CONDITION_ID != "G0":
    for group, name, _, _ in model_groups:
        settings[group][name]["retry"]["max_retries"] = (
            config.GRAPHRAG_VARIANT_MAX_RETRIES
        )
    settings["embedding_models"]["default_embedding_model"]["call_args"] = {
        "dimensions": config.EMBEDDING_DIMENSION
    }

## Standard indexing and page inputs

The native chunk size is 8191 `o200k_base` tokens, with zero overlap. It preserves
each prepared example page as one TextUnit. The baseline entity types are
organization, person, geo and event. Claims and prompt overrides remain disabled.
Local Search settings below bind the software defaults of GraphRAG 3.1.0.

In [5]:
settings.update(
    {
        "input": {
            "type": "jsonl",
            "file_pattern": "input_records.jsonl",
            "id_column": "input_record_id",
            "text_column": "text",
        },
        "input_storage": {"type": "file", "base_dir": "input"},
        "chunking": {
            "size": config.GRAPHRAG_CHUNK_SIZE,
            "overlap": config.GRAPHRAG_CHUNK_OVERLAP,
            "encoding_model": config.GRAPHRAG_ENCODING_MODEL,
        },
        "extract_graph": {"entity_types": condition["entity_types"]},
        "extract_claims": {"enabled": False},
        "vector_store": {"vector_size": config.EMBEDDING_DIMENSION},
        "local_search": config.GRAPHRAG_LOCAL_SEARCH,
    }
)
display(
    Markdown(
        tabulate(
            [
                ("GraphRAG", config.GRAPHRAG_VERSION),
                ("Index method", config.GRAPHRAG_INDEX_METHOD),
                ("Chunk tokens", config.GRAPHRAG_CHUNK_SIZE),
                ("Overlap", config.GRAPHRAG_CHUNK_OVERLAP),
                ("Entity types", ", ".join(condition["entity_types"])),
                ("Embedding dimensions", config.EMBEDDING_DIMENSION),
            ],
            headers=["Parameter", "Value"],
            tablefmt="github",
        )
    )
)
if condition["graph_source"] is not None:
    settings["workflows"] = config.GRAPHRAG_EMBED_ONLY_WORKFLOWS

| Parameter            | Value                            |
|----------------------|----------------------------------|
| GraphRAG             | 3.1.0                            |
| Index method         | standard                         |
| Chunk tokens         | 8191                             |
| Overlap              | 0                                |
| Entity types         | organization, person, geo, event |
| Embedding dimensions | 1536                             |

## Bind a new run before indexing

The run stores the selected input files, projection ledger and placeholder settings.
Input or model-configuration changes require a new folder. A digest also binds the
Azure endpoint, API version and deployment names; API-key rotation does not change it.

In [6]:
if RUN_AZURE_INDEX:
    assert azure_dir.resolve().is_relative_to(Path("../local").resolve())
    (azure_dir / "input").mkdir(parents=True, exist_ok=True)
    configuration = {
        "graphrag_version": config.GRAPHRAG_VERSION,
        "settings": settings,
        "index_method": config.GRAPHRAG_INDEX_METHOD,
        "question_set_version": input_manifest["question_set_version"],
    }
    if CONDITION_ID != "G0":
        configuration["condition_id"] = CONDITION_ID
        configuration["chunking_policy"] = condition["chunking_policy"]
    binding = {
        key: os.environ[key]
        for key in (
            "AZURE_OPENAI_ENDPOINT",
            "AZURE_OPENAI_API_VERSION",
            "AZURE_OPENAI_CHAT_DEPLOYMENT",
            embedding_variable,
        )
    }
    configuration["azure_binding_sha256"] = hashlib.sha256(
        json.dumps(binding, sort_keys=True).encode("utf-8")
    ).hexdigest()
    files = {
        "chunks.jsonl": chunks,
        "chunk_origins.jsonl": origins,
        "questions.jsonl": questions,
        "input/input_records.jsonl": inputs,
        "input_projection.jsonl": ledger,
    }
    for filename, records in files.items():
        path = azure_dir / filename
        content = "".join(json.dumps(row, ensure_ascii=False) + "\n" for row in records)
        if path.exists():
            assert path.read_text(encoding="utf-8") == content, (
                "Select a new run folder for changed inputs."
            )
        path.write_text(content, encoding="utf-8")
    text = json.dumps(configuration, ensure_ascii=False, indent=2) + "\n"
    configuration_path = azure_dir / "configuration.json"
    if configuration_path.exists():
        assert configuration_path.read_text(encoding="utf-8") == text, (
            "Select a new run folder for changed settings."
        )
    configuration_path.write_text(text, encoding="utf-8")
    (azure_dir / "settings.yaml").write_text(
        yaml.safe_dump(settings, sort_keys=False), encoding="utf-8"
    )

## Run the official index command

Only this selected generation mode makes model requests. The native file cache
reuses completed model work if an interrupted unchanged run is repeated.
A completed bound index is reused directly. CLI output stays in an ignored log.

### Reuse graph tables for EMBED and GraphRAG FINAL

`G-EMBED-LARGE-1536` retains the six native G0 tables. `G-FINAL` retains them
from the completed G-ENTITY index. Only `generate_text_embeddings` runs in these
conditions. Select that source with `source_index_dir`; the new run uses a distinct
output folder and vector store. Table bytes must remain unchanged after indexing.


In [7]:
if RUN_AZURE_INDEX and condition["graph_source"] is not None:
    source_manifest = json.loads(
        (source_index_dir / "index_manifest.json").read_text(encoding="utf-8")
    )
    assert source_manifest["condition_id"] == condition["graph_source"]
    assert source_manifest["graphrag_version"] == config.GRAPHRAG_VERSION
    source_paths = verify_artifacts(source_index_dir, source_manifest)
    assert (
        source_paths["chunks"].read_bytes() == (azure_dir / "chunks.jsonl").read_bytes()
    )
    assert (
        source_paths["input_projection"].read_bytes()
        == (azure_dir / "input_projection.jsonl").read_bytes()
    )
    assert source_index_dir.resolve() != azure_dir.resolve()
    (azure_dir / "output").mkdir(exist_ok=True)
    for name in (
        "documents",
        "text_units",
        "entities",
        "relationships",
        "communities",
        "community_reports",
    ):
        target = azure_dir / "output" / f"{name}.parquet"
        if target.exists():
            assert target.read_bytes() == source_paths[name].read_bytes()
        else:
            shutil.copyfile(source_paths[name], target)

In [8]:
if RUN_AZURE_INDEX:
    completed_manifest = azure_dir / "index_manifest.json"
    if completed_manifest.exists():
        saved = json.loads(completed_manifest.read_text(encoding="utf-8"))
        verify_artifacts(azure_dir, saved)
    else:
        command = [
            sys.executable,
            "-m",
            "graphrag",
            "index",
            "--root",
            str(azure_dir.resolve()),
            "--method",
            config.GRAPHRAG_INDEX_METHOD,
        ]
        if CONDITION_ID != "G0":
            command.append("--skip-validation")
        with (azure_dir / "index_cli.log").open("w", encoding="utf-8") as log:
            result = subprocess.run(
                command, stdout=log, stderr=subprocess.STDOUT, check=False
            )
        assert result.returncode == 0, (
            "Native indexing failed; inspect the ignored index_cli.log."
        )
    table_root = azure_dir / "output"
else:
    table_root = bound_paths["documents"].parent

## Inspect the native artefacts

Documents and TextUnits retain source text. Entities and relationships describe
the extracted graph. Communities group it; community reports summarise those groups.
The same native tables supply Local Search in the next notebook.

In [9]:
table_names = (
    "documents",
    "text_units",
    "entities",
    "relationships",
    "communities",
    "community_reports",
)
tables = {name: pd.read_parquet(table_root / f"{name}.parquet") for name in table_names}
documents, text_units = tables["documents"], tables["text_units"]
assert documents["id"].is_unique and text_units["human_readable_id"].is_unique
assert (
    set(documents["id"])
    == set(text_units["document_id"])
    == {row["input_record_id"] for row in inputs}
)
input_text = {row["input_record_id"]: row["text"] for row in inputs}
for row in text_units.itertuples(index=False):
    assert row.text == input_text[row.document_id]
assert len(text_units) == len(inputs), (
    "Baseline page inputs must stay as one TextUnit each."
)
display(
    Markdown(
        tabulate(
            [(name, len(table)) for name, table in tables.items()],
            headers=["Native table", "Rows"],
            tablefmt="github",
        )
    )
)
if RUN_AZURE_INDEX and condition["graph_source"] is not None:
    for name in table_names:
        assert (table_root / f"{name}.parquet").read_bytes() == source_paths[
            name
        ].read_bytes()
    verify_artifacts(source_index_dir, source_manifest)

| Native table      |   Rows |
|-------------------|--------|
| documents         |     20 |
| text_units        |     20 |
| entities          |    137 |
| relationships     |    216 |
| communities       |     20 |
| community_reports |     20 |

In [10]:
display(
    Markdown(
        tabulate(
            tables["entities"][["human_readable_id", "title", "type"]]
            .head(5)
            .values.tolist(),
            headers=["Entity ID", "Entity", "Type"],
            tablefmt="github",
        )
    )
)
display(
    Markdown(
        tabulate(
            tables["relationships"][["source", "target", "weight"]]
            .head(5)
            .values.tolist(),
            headers=["Source entity", "Target entity", "Weight"],
            tablefmt="github",
        )
    )
)

|   Entity ID | Entity                                | Type         |
|-------------|---------------------------------------|--------------|
|           0 | NETWORK WORKING GROUP                 | ORGANIZATION |
|           1 | S. CHRISTEY                           | PERSON       |
|           2 | MONKEYSEED, INC.                      | ORGANIZATION |
|           3 | INTERNET SOCIETY                      | ORGANIZATION |
|           4 | INFINITE MONKEY PROTOCOL SUITE (IMPS) | EVENT        |

| Source entity                         | Target entity                         |   Weight |
|---------------------------------------|---------------------------------------|----------|
| S. CHRISTEY                           | NETWORK WORKING GROUP                 |        8 |
| MONKEYSEED, INC.                      | INFINITE MONKEY PROTOCOL SUITE (IMPS) |        7 |
| INTERNET SOCIETY                      | NETWORK WORKING GROUP                 |        8 |
| INFINITE MONKEY PROTOCOL SUITE (IMPS) | WILLIAM SHAKESPEARE                   |        6 |
| INFINITE MONKEY PROTOCOL SUITE (IMPS) | INTERNET COMMUNITY                    |        1 |

## Retain the complete index binding

The manifest binds the six native Parquet tables and every native vector-store
file alongside inputs, settings and source mapping. Cache, logs and resolved secret
configuration remain outside the public example and exported datasets.

In [11]:
if RUN_AZURE_INDEX:
    paths = {
        name: azure_dir / filename
        for name, filename in (
            ("chunks", "chunks.jsonl"),
            ("chunk_origins", "chunk_origins.jsonl"),
            ("questions", "questions.jsonl"),
            ("graph_inputs", "input/input_records.jsonl"),
            ("input_projection", "input_projection.jsonl"),
            ("configuration", "configuration.json"),
            ("settings", "settings.yaml"),
        )
    }
    paths.update({name: table_root / f"{name}.parquet" for name in table_names})
    for path in sorted((table_root / "lancedb").rglob("*")):
        if path.is_file():
            paths["vector_file:" + path.relative_to(table_root).as_posix()] = path
    index_manifest = {
        "schema_version": "graphrag-index-artifacts-v1",
        "status": "completed",
        "graphrag_version": config.GRAPHRAG_VERSION,
        "run_id": azure_dir.name,
        "condition_id": CONDITION_ID,
        "question_set_version": input_manifest["question_set_version"],
        "artifacts": {
            role: {
                "path": path.relative_to(azure_dir).as_posix(),
                "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
            }
            for role, path in paths.items()
        },
    }
    if CONDITION_ID != "G0":
        index_manifest["chunking_policy"] = condition["chunking_policy"]
    if condition["graph_source"] is not None:
        index_manifest["graph_source"] = {
            "condition_id": source_manifest["condition_id"],
            "index_manifest_sha256": hashlib.sha256(
                (source_index_dir / "index_manifest.json").read_bytes()
            ).hexdigest(),
        }
    (azure_dir / "index_manifest.json").write_text(
        json.dumps(index_manifest, indent=2) + "\n", encoding="utf-8"
    )
    verify_artifacts(azure_dir, index_manifest)
    print({"index_status": "completed", "native_tables": len(table_names)})

## Native Local Search

[Local Search and evidence projection](03_graphrag_local_search.ipynb) reads this
index, displays its mixed context and original response, and maps Sources back to
the shared PDF pages. A new query can use the stored index without reindexing.